# Lesson 12 · Pooling & classic CNN architectures: computing LeNet, AlexNet and VGG yourself, then hitting the wall of depth

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/12-kien-truc-cnn-en.ipynb)

This notebook accompanies [Lesson 12 of Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/kien-truc-cnn/), a deep-learning course originally in
Vietnamese. It is the English edition of the course's notebook
([Vietnamese original](https://github.com/trungnguyenvn/hoc-sau-notebooks/blob/main/12-kien-truc-cnn.ipynb)):
the code, the data and every check are the same. The interactive demo is on the lesson page.
A complete CNN alternates convolution, ReLU and pooling to shrink the image, then ends with a classification layer.
Here we write each of its pieces ourselves in numpy, check each formula by computing directly, train a LeNet on real
MNIST, then recompute every layer of three classic architectures and compare with the published numbers.

**What you'll do:**
1. Write max pooling and average pooling, and see how they react when the image shifts by 1 or 2 pixels.
2. Check the output size formula and the receptive field formula by counting directly.
3. Train a LeNet on MNIST and measure how many predictions a 1-pixel shift of the image changes.
4. Count the parameters and multiply-accumulate operations of LeNet‑5, AlexNet and VGG‑16 layer by layer, and match
   the numbers in the papers.
5. Read the results of “plain” networks of 8–56 layers and see that the deeper networks learn worse: the wall of
   depth.

**Background:** [Lesson 11 · Convolutional networks (CNNs)](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/cnn/)
(convolution, shared filters), [Lesson 06 · Backpropagation](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/lan-truyen-nguoc/)
(used in section 6) and [Lesson 10 · BatchNorm, LayerNorm & RMSNorm](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/chuan-hoa/)
(the deep networks in section 10 have BatchNorm). **Run time:** under a minute on a CPU, including the first MNIST
download (~11 MB).

Every number in the notebook is computed by the cells below, and matches the numbers in the demo. The one exception
is the 3-seed training results of the 6 LeNet networks (5 epochs) and of the 45 deep networks, which are loaded from
the very files the demo uses, because rerunning them takes tens of minutes; the notebook trains a shorter LeNet itself
to show the mechanism.

> The **🤔 Predict first** cells ask a question before you run something. Answer it yourself, then run the next cell.

## 1 · Setup

Only numpy and matplotlib are needed (Colab has them preinstalled). The function `num()` prints numbers in the
English format: a decimal point, and commas between thousands. That is the same point Python uses when you *type* a
decimal (`0.1`); printing only adds the thousands separators and a real minus sign.

In [ ]:
import gzip
import hashlib
import json
import time
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter
from numpy.lib.stride_tricks import sliding_window_view

# The course's palette (colorblind-safe), always paired with a second encoding (marker shape, dash style, label)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
F32 = np.float32


def num(x, digits=2):
    """A number in the English format: 13,007 · 0.1 · −4.38."""
    return f"{x:,.{digits}f}".replace("-", "−")


def pct(p, digits=2):
    return num(100 * p, digits) + "%"


def tick(v, _pos=None):
    """An axis tick label: 0.5 · −2 · 60,000 (no trailing zeros)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return num(v, len(s.split(".")[1]) if "." in s else 0)


def fmt_axes(fig, which="xy"):
    """Put the tick labels of the figure's numeric axes in the English format; a log axis gets ticks at 1–2–5. Call
    right before plt.show(). which="x" when the y axis carries text labels (network names…), to keep those labels."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.axis_name not in which:
                continue
            if axis.get_scale() == "log":
                lo, hi = sorted(lim)
                ticks = [m * 10.0**k for k in range(-6, 4) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
                axis.set_major_locator(FixedLocator(ticks))
                axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(tick))


print(num(13007, 0), num(0.1, 1), num(-4.3805, 4), pct(0.993527))

## 2 · The data

MNIST (LeCun et al., 1998): 60,000 training images and 10,000 test images, grayscale 28×28 handwritten digits.
The cell below downloads the 4 original files and checks their MD5 hashes, so everyone who runs it gets exactly the
same data. The images are kept in the shape (images, height, width, channels) = (N, 28, 28, 1), with values in
[0, 1], as float32, like the demo's scripts.

The cell also loads the two results files the demo uses (`fetch_json`, with an MD5 check): 6 ways for LeNet to shrink
the image × 3 seeds, and the “plain” networks of 8–56 layers × 3 learning rates × 3 seeds.

In [ ]:
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {  # MNIST's standard MD5 hashes
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}
DATA = Path("mnist")


def fetch():
    DATA.mkdir(exist_ok=True)
    for name, md5 in FILES.items():
        p = DATA / name
        if not p.exists():
            urllib.request.urlretrieve(URL + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"


def load(kind):
    """IDX format: the first 16 bytes of an image file (8 bytes of a label file) are the header; the rest are bytes
    0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        images = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 28, 28)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        labels = np.frombuffer(f.read(), np.uint8, offset=8)
    return images, labels


# The results of the long training runs (the same files as the demo on the website). build.py replaces COURSE_DATA_URL
# with the data/ folder of the public notebook repo.
def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"
    return json.loads(p.read_text())


fetch()
lenet_runs = fetch_json("mnist-lenet.json", "bf1680c82271d41115500fb24e4e6f8f")  # 6 ways to shrink × 3 seeds
depth_runs = fetch_json("mnist-depth.json", "8156284d0611a3445408647bbae5d813")  # plain networks of 8–56 layers

In [ ]:
train_img, train_lab = load("train")
test_img, test_lab = load("t10k")
Xtr = (train_img / 255.0).astype(F32)[..., None]  # (60,000, 28, 28, 1)
Xte = (test_img / 255.0).astype(F32)[..., None]
ytr, yte = train_lab.astype(np.int64), test_lab.astype(np.int64)
print("training:", Xtr.shape, " test:", Xte.shape, " dtype:", Xtr.dtype)
assert (len(ytr), len(yte)) == (lenet_runs["nTrain"], lenet_runs["nTest"])
n_deep = sum(len(r["seeds"]) for r in depth_runs["runs"])
print(
    f"LeNet file: {len(lenet_runs['runs'])} networks × {len(lenet_runs['seeds'])} seeds · "
    f"depth file: {n_deep} deep networks"
)

One image of each digit, taken from the test set:

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(10, 1.4))
for d, ax in enumerate(axes):
    ax.imshow(Xte[np.flatnonzero(yte == d)[0], :, :, 0], cmap="gray_r")
    ax.set_title(str(d), fontsize=11)
    ax.axis("off")
plt.show()

## 3 · Max pooling shrinks the feature map

Pooling divides the map into $k \times k$ windows and replaces each window with a single number: **max pooling**
keeps the largest value, **average pooling** takes the mean. With $k = 2$ and stride $s = 2$, each side is halved,
and the number of values drops by a factor of 4. Pooling has no weights to learn.

We start with a one-dimensional signal and a loop. The function `vec()` prints a sequence of numbers in the English
format.

In [ ]:
def vec(a):
    return "[" + "  ".join(tick(v) for v in a) + "]"


def pool1d_loop(x, k=2, s=2, kind="max"):
    out = []
    for start in range(0, len(x) - k + 1, s):  # the windows x[start : start + k]
        window = x[start : start + k]
        out.append(max(window) if kind == "max" else sum(window) / k)
    return np.array(out)


x1 = np.array([0, 0, 1, 1, 0, 0, 1, 1, 0, 0], float)
print("x                 =", vec(x1))
print("max pooling 2     =", vec(pool1d_loop(x1)))
print("average pooling 2 =", vec(pool1d_loop(x1, kind="avg")))

10 values become 5: each pair $(x_1, x_2), (x_3, x_4), \dots$ becomes one number.

### 🤔 Predict first
Shift $x$ to the right by **1 position** (fill the empty position with 0). What does max pooling give: still
$[0\ 1\ 0\ 1\ 0]$, a shifted copy of it, or something else entirely? And for a shift of **2 positions**?

In [ ]:
def shift1d(x, d):
    """Shift the signal right by d positions, filling the gap with 0."""
    return np.concatenate([np.zeros(d), x[: len(x) - d]])


for d in (0, 1, 2):
    xs = shift1d(x1, d)
    print(f"shift {d}: x = {vec(xs)}  →  max {vec(pool1d_loop(xs))}   average {vec(pool1d_loop(xs, kind='avg'))}")
same2 = np.array_equal(pool1d_loop(shift1d(x1, 2)), shift1d(pool1d_loop(x1), 1))
print("shift 2 then pool = pool then shift 1:", same2)
assert same2

Shifted by 1 position, max pooling gives $[0\ 1\ 1\ 1\ 1]$: the two separate peaks become one continuous band. That
is neither the old output nor a shifted copy of it. Average pooling gives $[0\ 0.5\ 0.5\ 0.5\ 0.5]$, which loses the
shape entirely too. Only when the shift is exactly **a multiple of the stride** (2 positions) does the output shift
along with it, by 1 position.

Zhang (2019) explains this by splitting max pooling into two steps: take the max over **every** window (stride 1, so
shifting the input shifts the output along with it), then **subsample**, keeping one cell in two. The second step
destroys that property, because it ignores the sampling theorem (Nyquist): a signal that changes faster than the
sampling rate gets “folded” into a different signal (aliasing). The classic fix in image processing is to blur before
subsampling; Zhang calls the version for max pooling **MaxBlurPool**: a 2×2 max with stride 1, a blur with the filter
$[1\ 2\ 1]/4$ along each dimension, and only then stride 2.

On images, we write 2×2 pooling by taking the 4 corners of every 2×2 window (the four arrays `x[:, i::2, j::2]`).
With an odd side, the last row/column is dropped.

In [ ]:
QUAD = [(0, 0), (0, 1), (1, 0), (1, 1)]  # the 4 positions in a 2×2 window


def pool2(x, kind="max"):
    """2×2 pooling with stride 2 over axes 1 and 2 of x, (N, H, W) or (N, H, W, C)."""
    H, W = x.shape[1] // 2 * 2, x.shape[2] // 2 * 2  # odd side: drop the last row/column
    q = [x[:, i:H:2, j:W:2] for i, j in QUAD]
    if kind == "avg":
        return (q[0] + q[1] + q[2] + q[3]) / 4
    return np.maximum(np.maximum(q[0], q[1]), np.maximum(q[2], q[3]))


def subsample2(x):
    """Subsampling only (one cell in two along each dimension), like a stride-2 convolution without a filter."""
    return x[:, ::2, ::2]


def maxblurpool2(x):
    """Zhang (2019): 2×2 max with stride 1, a 3×3 blur [1 2 1]/4 ⊗ [1 2 1]/4, then stride 2 (like
    convnet.MaxBlurPool)."""
    H, W = x.shape[1:3]
    rest = [(0, 0)] * (x.ndim - 3)
    xp = np.pad(x, [(0, 0), (0, 1), (0, 1), *rest], mode="edge")  # keeps the size H × W after the stride-1 max
    m = np.maximum(np.maximum(xp[:, :H, :W], xp[:, 1:, :W]), np.maximum(xp[:, :H, 1:], xp[:, 1:, 1:]))
    mp = np.pad(m, [(0, 0), (1, 1), (1, 1), *rest], mode="reflect")
    w = np.array([1, 2, 1]) / 4
    out = sum(w[i] * w[j] * mp[:, i : i + H, j : j + W] for i in range(3) for j in range(3))
    return out[:, ::2, ::2]


def shift(X, dx, dy):
    """Shift the digit dx pixels to the right and dy pixels down, filling with 0 (like the demo's convnet.shift)."""
    out = np.zeros_like(X)
    H, W = X.shape[1:3]
    ys, yd = (slice(0, H - dy), slice(dy, H)) if dy >= 0 else (slice(-dy, H), slice(0, H + dy))
    xs, xd = (slice(0, W - dx), slice(dx, W)) if dx >= 0 else (slice(-dx, W), slice(0, W + dx))
    out[:, yd, xd] = X[:, ys, xs]
    return out


digit = Xte[:1, :, :, 0].astype(float)  # the first test image, (1, 28, 28)
for name, f in [("max pooling", pool2), ("average pooling", lambda x: pool2(x, "avg")), ("MaxBlurPool", maxblurpool2)]:
    print(f"{name:>15}: {digit.shape[1:]} → {f(digit).shape[1:]}")

The figure below applies the four ways of shrinking to a real digit (top row) and to the same digit shifted 1 pixel to
the right (bottom row). The first column is the 28×28 image; the next four columns are the 14×14 versions:
subsampling only (one cell in two, like a stride-2 convolution), max pooling, average pooling and MaxBlurPool. The
darker the color, the larger the value.

In [ ]:
DOWN = {
    "subsampling": subsample2,
    "max pooling": pool2,
    "average pooling": lambda x: pool2(x, "avg"),
    "MaxBlurPool": maxblurpool2,
}
j_demo = np.flatnonzero(yte == 7)[0]
pair = np.concatenate([Xte[j_demo : j_demo + 1, :, :, 0], shift(Xte[j_demo : j_demo + 1, :, :, 0], 1, 0)])
fig, axes = plt.subplots(2, 5, figsize=(10, 5.0))
for r, row_name in enumerate(["original", "shifted 1 pixel →"]):
    axes[r, 0].imshow(pair[r], cmap="gray_r", vmin=0, vmax=1)
    axes[r, 0].set_title(f"{row_name}\n28×28", fontsize=10)
    for c, (name, f) in enumerate(DOWN.items(), start=1):
        axes[r, c].imshow(f(pair[r : r + 1])[0], cmap="gray_r", vmin=0, vmax=1)
        axes[r, c].set_title(f"{name}\n14×14", fontsize=10)
for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():  # a gray frame on all 4 sides: the image's edge stays visible when it shifts
        sp.set_visible(True)
        sp.set_color(GREY)
plt.tight_layout()
plt.show()

In the bottom row, each 14×14 version changes in its own way rather than just sliding: a thin vertical stroke can
fall into a different window of the 2×2 grid, so it comes out darker or lighter. The cell below measures this on all
10,000 test images, with the relative change $\lVert f(\text{shift}(x)) - f(x) \rVert / \lVert f(x) \rVert$:
- **1-pixel shift:** compared with the output for the original image;
- **2-pixel shift:** compared with the output for the original image shifted by 1 cell (14×14), that is, asking
  “does the output shift by exactly 1 cell?”.

In [ ]:
X2 = Xte[:, :, :, 0].astype(float)  # 10,000 images (N, 28, 28)


def rel_change(a, b):
    """The mean over the images of ‖a − b‖ / ‖b‖."""
    num = np.linalg.norm((a - b).reshape(len(a), -1), axis=1)
    return float(np.mean(num / np.linalg.norm(b.reshape(len(b), -1), axis=1)))


sens = {}
for name, f in DOWN.items():
    base = f(X2)
    sens[name] = (rel_change(f(shift(X2, 1, 0)), base), rel_change(f(shift(X2, 2, 0)), shift(base, 1, 0)))
    print(
        f"{name:>15}: 1-pixel shift → change {pct(sens[name][0], 1):>6}   "
        f"2-pixel shift → deviation {num(sens[name][1], 4)}"
    )
assert all(sens[n][1] < 1e-12 for n in ("subsampling", "max pooling", "average pooling"))
assert sens["MaxBlurPool"][0] < min(sens[n][0] for n in ("subsampling", "max pooling", "average pooling"))

With a 2-pixel shift (a multiple of the stride), all three methods that apply stride 2 directly give exactly the old
output shifted by 1 cell: the deviation is 0. With a 1-pixel shift, no method keeps the output unchanged: a 1-pixel
step in the 28×28 image is half a cell in the 14×14 grid, and no grid can shift by half a cell. Max pooling and
average pooling change by almost the same amount (46.5% and 47.2%), less than subsampling. MaxBlurPool changes the
least, because on a blurred map two neighboring cells have similar values, but its change is not 0 either. Its very
small deviation for a 2-pixel shift comes from the border: the padding at the edge does not shift with the image.

Max pooling tolerates small shifts only **partly**: if the strongest stroke stays within the same 2×2 window, the
output does not change. Section 6 measures whether that is enough to keep the whole network's prediction unchanged.

## 4 · The output size formula

A convolutional or pooling layer slides a $k \times k$ window over a map of side $n_{\text{in}}$, padded with $p$
cells on each side, jumping $s$ cells at a time. The number of positions the window can take along each side is

$$n_{\text{out}} = \left\lfloor \frac{n_{\text{in}} + 2p - k}{s} \right\rfloor + 1$$

The fractional part of the division is rounded down: the last rows that are not enough for one more step are dropped,
and the filter never looks at them. The cell below checks the formula by *directly counting* the window's starting
positions, for every combination $n_{\text{in}} \le 40$, $k \in \{1, 2, 3, 5, 7, 11\}$, $s \le 4$, $p \le 3$, and
against the actual size of the array that the function `patches()` (used for convolution in section 6) creates.

In [ ]:
def n_out(n, k, s=1, p=0):
    return (n + 2 * p - k) // s + 1


def n_out_loop(n, k, s=1, p=0):
    """Direct count: the window starts at cell start (in the padded image's coordinates) and must fit entirely inside
    the padded image."""
    return sum(1 for start in range(0, n + 2 * p, s) if start + k <= n + 2 * p)


def patches(x, k, s=1, p=0):
    """(N, H, W, C) → (N, Ho, Wo, k·k·C): each output cell gets its k×k×C patch, in (row, column, channel) order."""
    if p:
        x = np.pad(x, ((0, 0), (p, p), (p, p), (0, 0)))
    w = sliding_window_view(x, (k, k), axis=(1, 2))[:, ::s, ::s]  # (N, Ho, Wo, C, k, k)
    N, Ho, Wo, C = w.shape[:4]
    return w.transpose(0, 1, 2, 4, 5, 3).reshape(N, Ho, Wo, k * k * C)


cases = [(n, k, s, p) for n in range(1, 41) for k in (1, 2, 3, 5, 7, 11) for s in (1, 2, 3, 4) for p in (0, 1, 2, 3)]
cases = [c for c in cases if c[0] + 2 * c[3] >= c[1]]  # the filter must fit in the padded image
ok = all(n_out(*c) == n_out_loop(*c) for c in cases)
ok_shape = all(patches(np.zeros((1, n, n, 1), F32), k, s, p).shape[1] == n_out(n, k, s, p) for n, k, s, p in cases)
print(
    f"{num(len(cases), 0)} combinations (n, k, s, p): formula = direct count: {ok} · "
    f"= array size from patches(): {ok_shape}"
)
assert ok and ok_shape

A few familiar cases, with the number of rows dropped by rounding down, $(n_{\text{in}} + 2p - k) \bmod s$:

In [ ]:
for label, (n, k, s, p) in [
    ("3×3 convolution, p = 1 (same size)", (28, 3, 1, 1)),
    ("2×2 max pooling, stride 2", (28, 2, 2, 0)),
    ("2×2 max pooling on an odd side", (7, 2, 2, 0)),
    ("AlexNet layer 1: 11×11 stride 4, image 224", (224, 11, 4, 0)),
    ("AlexNet layer 1: 11×11 stride 4, image 227", (227, 11, 4, 0)),
]:
    print(f"{label:<42} ⌊({n} + 2·{p} − {k}) / {s}⌋ + 1 = {n_out(n, k, s, p):>3}   rows dropped: {(n + 2 * p - k) % s}")
assert n_out(227, 11, 4) == 55 and (227 - 11) % 4 == 0 and (224 - 11) % 4 == 1 and n_out(224, 11, 4, 2) == 55

The AlexNet paper says the input image is 224×224 and says nothing about padding. Without padding, with an 11×11
filter and stride 4, $(224 - 11)/4$ does not divide evenly: it gives 54 cells and the last row is dropped, while the
paper draws a 55×55 map. A 227×227 image gives exactly 55 (CS231n pointed this out); a 224 image padded by 2 cells on
each side also gives 55, because $\lfloor (224 + 4 - 11)/4 \rfloor + 1 = 55$, and that is exactly what torchvision's
AlexNet does. 2×2 pooling on an odd side of 7 also drops the last row: it gives 3 cells, and the 7th row belongs to no
window.

## 5 · The receptive field grows with depth

The **receptive field** of a cell is the region of the input image that can affect that cell's value. A
$k \times k$ layer with stride 1 widens that region by $k - 1$ pixels. After a layer with stride $s$, two neighboring
cells are $s$ times farther apart on the input image, so each later layer widens the region more. Let $j$ be the
distance (in pixels of the input image) between two neighboring cells. Starting from $r = 1$, $j = 1$, through each
layer:

$$r \leftarrow r + (k - 1)\, j, \qquad j \leftarrow j \cdot s$$

(Araujo et al., 2019). Padding does not change the size of the region, it only shifts its position. We describe a
network by a list of layers; the function `rf()` runs the formula above. The demo's LeNet has two 5×5 convolutional
layers and two 2×2 poolings.

In [ ]:
LENET_SPATIAL = [("conv", 5, 1, 2), ("pool", 2, 2, 0), ("conv", 5, 1, 0), ("pool", 2, 2, 0)]  # (kind, k, s, p)


def rf(layers):
    """The receptive field r, the distance j between two adjacent cells, and the effective padding P (Araujo et al.,
    2019)."""
    r, j, P = 1, 1, 0
    for _, k, s, p in layers:
        r, P, j = r + (k - 1) * j, P + p * j, j * s
    return r, j, P


for depth in range(1, len(LENET_SPATIAL) + 1):
    name, k, s, p = LENET_SPATIAL[depth - 1]
    r, j, _ = rf(LENET_SPATIAL[:depth])
    print(
        f"after layer {depth} ({k}×{k} {'convolution' if name == 'conv' else 'pooling'}, stride {s}): "
        f"receptive field {r}×{r}, j = {j}"
    )

A direct check, without the formula: feed in 784 images, each with only **one** pixel lit, through exactly the 4
layers above with every weight equal to 1 (convolution = the sum of the window, pooling = max). The input values are
all nonnegative, so an output cell is nonzero if and only if the lit pixel lies in its receptive field. According to
Araujo et al., cell $u$ (counting from 0) sees from pixel $u \cdot j - P$ to $u \cdot j - P + r - 1$, where $P$ is
the effective padding that `rf()` returns.

In [ ]:
def influence(layers, n=28):
    """For each input pixel: which cells of the last layer does it influence? → (n·n, n_out, n_out), bool."""
    x = np.eye(n * n, dtype=F32).reshape(n * n, n, n, 1)  # image i lights only pixel i
    for kind, k, s, p in layers:
        win = patches(x, k, s, p)  # (N, Ho, Wo, k·k)
        x = (win.sum(-1) if kind == "conv" else win.max(-1))[..., None]
    return x[..., 0] > 0


r, j, P = rf(LENET_SPATIAL)
hit = influence(LENET_SPATIAL)
print(f"last layer: {hit.shape[1]}×{hit.shape[2]} cells; formula: r = {r}, j = {j}, P = {P}")
for u in range(hit.shape[1]):
    mask = hit[:, u, u].reshape(28, 28)
    rows = np.flatnonzero(mask.any(1))
    lo, hi = u * j - P, u * j - P + r - 1
    print(
        f"cell ({u}, {u}): influenced by rows {rows.min()} to {rows.max()}; formula: {num(lo, 0)} to {hi} "
        "(clipped to the image, 0…27)"
    )
    box = np.zeros((28, 28), bool)
    box[max(lo, 0) : hi + 1, max(lo, 0) : hi + 1] = True
    assert np.array_equal(mask, box), "the directly counted receptive field differs from the formula"

The direct count matches the formula at every cell. The middle cell, (2, 2), sees exactly a 16×16 region; the corner
cell, (0, 0), has a region starting at row −2, so 2 of its rows and 2 of its columns fall in the zero padding. The
figure below draws those two regions on a digit: the blue shading marks the pixels the direct count found, the dashed
frame is the $r \times r$ region given by the formula (the corner cell's frame sticks out of the image, into the
padding).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7, 3.6))
for ax, u in zip(axes, (0, 2)):
    mask = hit[:, u, u].reshape(28, 28)
    ax.imshow(Xte[j_demo, :, :, 0], cmap="gray_r", vmin=0, vmax=1)
    ax.imshow(np.where(mask, 1.0, np.nan), cmap=LinearSegmentedColormap.from_list("x", [BLUE, BLUE]), alpha=0.35)
    lo = u * j - P
    ax.add_patch(Rectangle((lo - 0.5, lo - 0.5), r, r, fill=False, ls="--", lw=2, ec="black", clip_on=False))
    ax.set(xlim=(-3.5, 27.5), ylim=(27.5, -3.5), xticks=[], yticks=[])
    ax.set_title(f"cell ({u}, {u}) after layer 4: region {r}×{r}", fontsize=10)
plt.tight_layout()
plt.show()

### Two 3×3 layers or one 5×5 layer?

By the formula, two 3×3 layers with stride 1 have a receptive field of $1 + 2 + 2 = 5$, the same as one 5×5 layer;
three 3×3 layers equal one 7×7 layer. But the number of weights differs. With $C$ input channels and $C$ output
channels, a $k \times k$ layer has $k^2 C^2$ weights (ignoring the biases), so:

In [ ]:
for n_layers, k_big in [(2, 5), (3, 7)]:
    r_stack, _, _ = rf([("conv", 3, 1, 1)] * n_layers)
    w_stack, w_big = n_layers * 3 * 3, k_big * k_big
    print(
        f"{n_layers} 3×3 layers: receptive field {r_stack}×{r_stack}, {w_stack}C² weights · 1 {k_big}×{k_big} layer: "
        f"{w_big}C²  ({pct(w_big / w_stack - 1, 0)} more) · with C = 512: {num(w_stack * 512**2, 0)} vs "
        f"{num(w_big * 512**2, 0)}"
    )
    assert r_stack == k_big
assert (2 * 9, 25, 3 * 9, 49) == (18, 25, 27, 49) and round(100 * (49 / 27 - 1)) == 81

That is why VGG (Simonyan & Zisserman, 2014) uses only 3×3 filters: the same field of view, fewer weights, and an
extra ReLU between every two layers. The paper gives exactly this number: a 7×7 layer needs $49C^2$ parameters, “81%
more” than the $27C^2$ of three 3×3 layers.

### The receptive fields of VGG‑16 and AlexNet
VGG‑16 uses only 3×3 filters, together with 5 rounds of 2×2 max pooling. Small filters do not mean a narrow view:
each pooling doubles $j$, so the deep 3×3 layers widen the region very quickly. The cell below computes the
receptive field of one cell in the 7×7×512 map after the last pooling, and that of AlexNet with its large 11×11 and
5×5 filters.

In [ ]:
VGG16_SPATIAL = []
for n_conv in (2, 2, 3, 3, 3):  # 5 blocks, each a few 3×3 layers followed by 2×2 max pooling
    VGG16_SPATIAL += [("conv", 3, 1, 1)] * n_conv + [("pool", 2, 2, 0)]
ALEXNET_SPATIAL = [("conv", 11, 4, 0), ("pool", 3, 2, 0), ("conv", 5, 1, 2), ("pool", 3, 2, 0)]
ALEXNET_SPATIAL += [("conv", 3, 1, 1)] * 3 + [("pool", 3, 2, 0)]
for name, net in [("VGG‑16", VGG16_SPATIAL), ("AlexNet", ALEXNET_SPATIAL)]:
    r_, j_, P_ = rf(net)
    print(
        f"{name:>8}: {len(net)} spatial layers → receptive field {r_}×{r_}, step between two cells j = {j_}, "
        f"padding P = {P_}"
    )
# Araujo et al. (2019), table of common networks: vgg_16 (r, S, P) = (212, 32, 90), alexnet_v2 = (195, 32, 64)
# (TF-Slim's alexnet_v2 has the same spatial layers: k, stride and padding as above)
assert rf(VGG16_SPATIAL) == (212, 32, 90) and rf(ALEXNET_SPATIAL) == (195, 32, 64)

After 13 3×3 layers and 5 poolings, one cell of VGG‑16 sees a 212×212 region, nearly the whole 224×224 image.
AlexNet reaches 195×195 with 8 spatial layers. Both match the table of Araujo et al. (2019), which the cell above
uses as an `assert`. In both networks, the step between two adjacent cells is 32 pixels (VGG‑16: $2^5$ through 5
poolings; AlexNet: $4 \cdot 2 \cdot 2 \cdot 2$, the first layer's stride 4, then 3 poolings): the last map is a coarse
grid, and each cell sees a wide region.

The receptive field only tells you which pixels *can* affect a cell. A pixel in the middle of the region has more
paths to the cell than a pixel at the edge, so it has much more influence; Luo et al. (2016) call the part that
really matters the *effective receptive field*, and measured it to be roughly bell-shaped, much smaller than
$r \times r$.

## 6 · LeNet on MNIST: training and shifting the image

Now we put the pieces together. The demo's LeNet (for 28×28 images) consists of:

5×5 convolution × 6, padding 2 → ReLU → 2×2 pooling → 5×5 convolution × 16 → ReLU → 2×2 pooling → dense 400 → 120 →
84 → 10.

A convolution is a matrix multiplication: `patches()` gathers each cell's $k \times k \times C$ patch into one row,
which is then multiplied by a weight matrix $W$ of size $(k^2 C) \times C_{\text{out}}$, like the $z = xW + b$ of a
dense layer. The cell below checks this against an explicit loop on one image, then writes the backpropagation of
each layer. Max pooling sends the gradient back to exactly the largest cell of each 2×2 window (split equally if
several cells tie, as the demo's `convnet.Pool` does).

In [ ]:
def unpatches(d, shape, k, p=0):
    """The inverse of patches() (stride 1): add each row's gradient back onto its own k×k×C patch."""
    N, H, W, C = shape
    Ho, Wo = d.shape[1:3]
    d = d.reshape(N, Ho, Wo, k, k, C)
    dx = np.zeros((N, H + 2 * p, W + 2 * p, C), F32)
    for i in range(k):
        for j in range(k):
            dx[:, i : i + Ho, j : j + Wo] += d[:, :, :, i, j]
    return dx[:, p : p + H, p : p + W] if p else dx


def pool_fwd(x, kind):
    """2×2 pooling with stride 2 on (N, H, W, C), even sides; returns the output and what backpropagation needs."""
    y = pool2(x, kind)
    if kind == "avg":
        return y, None
    eq = [x[:, i::2, j::2] == y for i, j in QUAD]  # which cells are the max (several may tie)
    return y, (eq, eq[0].astype(F32) + eq[1] + eq[2] + eq[3])


def pool_bwd(dy, cache, kind):
    N, a, b, C = dy.shape
    dx = np.empty((N, 2 * a, 2 * b, C), F32)
    for t, (i, j) in enumerate(QUAD):
        dx[:, i::2, j::2] = dy / 4 if kind == "avg" else cache[0][t] * (dy / cache[1])
    return dx


# Check: convolution by a loop = patches() @ W, on the first test image
rng = np.random.default_rng(0)
W_try = rng.normal(0, 0.2, (5 * 5 * 1, 6)).astype(F32)
img = Xte[:1]
loop = np.zeros((28, 28, 6), F32)
padded = np.pad(img[0], ((2, 2), (2, 2), (0, 0)))
for u in range(28):
    for v in range(28):
        loop[u, v] = padded[u : u + 5, v : v + 5, :].reshape(-1) @ W_try  # 25 MACs per filter
print("loop = patches() @ W:", np.allclose(loop, (patches(img, 5, 1, 2) @ W_try)[0], atol=1e-5))
assert np.allclose(loop, (patches(img, 5, 1, 2) @ W_try)[0], atol=1e-5)

The complete network. He initialization $\mathcal{N}(0, 2/\text{fan-in})$ for every weight, biases at 0, drawn from
`default_rng(seed)` in exactly the same layer order as the demo's `lenet.py` script; each epoch shuffles the order of
the images with the same generator. Training uses SGD with $\eta = 0.1$, batches of 64 images, and the cross-entropy
loss (Lesson 04).

In [ ]:
SHAPES = {"W1": (25, 6), "W2": (150, 16), "W3": (400, 120), "W4": (120, 84), "W5": (84, 10)}  # (fan-in, fan-out)


def init_lenet(seed):
    rng = np.random.default_rng(seed)
    P = {}
    for name, (fan_in, fan_out) in SHAPES.items():
        P[name] = rng.normal(0, np.sqrt(2 / fan_in), (fan_in, fan_out)).astype(F32)
        P["b" + name[1:]] = np.zeros(fan_out, F32)
    return P, rng


def forward(P, x, kind="max", c=None):
    """x: (N, 28, 28, 1) → scores (N, 10). c (if given) keeps the intermediate values for backpropagation."""
    x1 = patches(x, 5, 1, 2)  # (N, 28, 28, 25)
    a1 = x1 @ P["W1"] + P["b1"]  # (N, 28, 28, 6)
    p1, m1 = pool_fwd(np.maximum(a1, 0), kind)  # (N, 14, 14, 6)
    x2 = patches(p1, 5)  # (N, 10, 10, 150)
    a2 = x2 @ P["W2"] + P["b2"]  # (N, 10, 10, 16)
    p2, m2 = pool_fwd(np.maximum(a2, 0), kind)  # (N, 5, 5, 16)
    f = p2.reshape(len(x), 400)
    h3 = np.maximum(f @ P["W3"] + P["b3"], 0)
    h4 = np.maximum(h3 @ P["W4"] + P["b4"], 0)
    if c is not None:
        c.update(x1=x1, a1=a1, m1=m1, x2=x2, a2=a2, m2=m2, f=f, h3=h3, h4=h4)
    return h4 @ P["W5"] + P["b5"]


def sgd_step(P, x, y, kind="max", eta=0.1):
    c = {}
    z = forward(P, x, kind, c)
    p = np.exp(z - z.max(1, keepdims=True))
    p /= p.sum(1, keepdims=True)
    p[np.arange(len(y)), y] -= 1
    d = (p / len(y)).astype(F32)  # ∂L/∂z of the mean cross-entropy
    g = {"W5": c["h4"].T @ d, "b5": d.sum(0)}
    d = (d @ P["W5"].T) * (c["h4"] > 0)
    g["W4"], g["b4"] = c["h3"].T @ d, d.sum(0)
    d = (d @ P["W4"].T) * (c["h3"] > 0)
    g["W3"], g["b3"] = c["f"].T @ d, d.sum(0)
    d = pool_bwd((d @ P["W3"].T).reshape(-1, 5, 5, 16), c["m2"], kind) * (c["a2"] > 0)
    g["W2"], g["b2"] = c["x2"].reshape(-1, 150).T @ d.reshape(-1, 16), d.sum((0, 1, 2))
    d = pool_bwd(unpatches(d @ P["W2"].T, (len(y), 14, 14, 6), 5), c["m1"], kind) * (c["a1"] > 0)
    g["W1"], g["b1"] = c["x1"].reshape(-1, 25).T @ d.reshape(-1, 6), d.sum((0, 1, 2))
    for k in P:
        P[k] -= eta * g[k]


def predict(P, X, kind="max"):
    return np.concatenate([forward(P, X[i : i + 1000], kind).argmax(1) for i in range(0, len(X), 1000)])


def train_lenet(kind="max", seed=0, steps=500, batch=64):
    P, rng = init_lenet(seed)
    order = rng.permutation(len(Xtr))  # epoch 1 of lenet.py: the same initial weights, the same image order
    for t in range(steps):
        idx = order[t * batch : (t + 1) * batch]
        sgd_step(P, Xtr[idx], ytr[idx], kind)
    return P


n_params = sum(v.size for v in init_lenet(0)[0].values())
print(f"LeNet for MNIST: {num(n_params, 0)} parameters")
assert n_params == 61706 == lenet_runs["runs"]["max"]["params"]

The demo trains this network for 5 epochs on all 60,000 images; with numpy that takes a few minutes. We run a
shorter version: the first 600 batches of epoch 1, with exactly the initial weights and image order of seed 0.

### 🤔 Predict first
The network has two max poolings. After training, we shift **every** test image by 1 pixel (right, left, down or up;
the gap is filled with 0). What percentage of the predictions change: 0%, a few percent, or more than half?

In [ ]:
STEPS = 600
t0 = time.perf_counter()
P_max = train_lenet("max", seed=0, steps=STEPS)
pred0 = predict(P_max, Xte)
print(f"{STEPS} SGD steps, {num(STEPS * 64, 0)} images ({num(time.perf_counter() - t0, 1)} seconds)")
print(f"{pct((pred0 == yte).mean())} of the test images correct")

DIRS = [(1, 0), (-1, 0), (0, 1), (0, -1)]  # (dx, dy): right, left, down, up
moved = {d: predict(P_max, shift(Xte, *d)) for d in DIRS}
changed = {d: moved[d] != pred0 for d in DIRS}
change1 = np.mean([changed[d].mean() for d in DIRS])  # averaged over the 4 directions, as in the demo
acc1 = np.mean([(moved[d] == yte).mean() for d in DIRS])
print(f"1-pixel shift: {pct(change1)} of the predictions change (average of 4 directions); {pct(acc1)} still correct")
print(
    " · ".join(f"{name} {pct(changed[d].mean())}" for name, d in zip(["right", "left", "down", "up"], DIRS)),
    f"· changed in at least one direction: {num(np.any(list(changed.values()), axis=0).sum(), 0)} images",
)
assert (pred0 == yte).mean() > 0.95 and change1 > 0.005

A few percent, not 0. Max pooling does not make the network **invariant** to shifts: as in section 3, a 1-pixel shift
is a shift of half a cell of the grid after the first pooling and a quarter of a cell after the second, so the
features don't just slide, they change value. For images that lie close to the boundary between two classes, that is
enough to change the answer. Azulay & Weiss (2018) measured something similar on ImageNet networks: shifting the
image down by 1 pixel changes up to 30% of the predictions, even though those networks were trained with data
augmentation.

The top row of the figure below shows the first 8 test images whose prediction changes when they are shifted right
by 1 pixel, and the bottom row shows them after the shift; the titles give the prediction, and the true label sits
under each image of the top row.

In [ ]:
flip = np.flatnonzero(changed[(1, 0)])[:8]
right = shift(Xte[flip], 1, 0)
fig, axes = plt.subplots(2, 8, figsize=(10, 3.1))
for c, i in enumerate(flip):
    for r, (img_, p) in enumerate([(Xte[i], pred0[i]), (right[c], moved[(1, 0)][i])]):
        axes[r, c].imshow(img_[:, :, 0], cmap="gray_r", vmin=0, vmax=1)
        axes[r, c].set_title(f"{'original' if r == 0 else 'shifted →'}: says {p}", fontsize=9)
        axes[r, c].set_xticks([])
        axes[r, c].set_yticks([])
    axes[0, c].set_xlabel(f"label {yte[i]}", fontsize=9)
plt.tight_layout()
plt.show()

## 7 · The full results: six ways of shrinking, 3 seeds

The run above is short (600 steps) and uses only one seed. The demo's `lenet.py` script trains the same network for
the full 5 epochs on 60,000 images with seeds 0, 1 and 2, for six variants:
- **max pooling**, **average pooling**, **MaxBlurPool**: the three kinds of pooling from section 3;
- **stride 2**: no pooling, but two convolutional layers with stride 2, so the map still shrinks (subsampling only,
  no aggregation);
- **no pooling**: the map keeps a size of 24×24, so the first dense layer receives 9,216 values;
- **Lesson 11**: the previous lesson's one-layer CNN (a 3×3 convolution × 8, one max pooling).

Each network is tested on test images shifted by $k = 1 \dots 4$ pixels, averaged over the four directions as in
section 6. Rerunning this takes a few minutes, so we read the results from the demo's file. Each number below is the
min–max range over the 3 seeds.

In [ ]:
VARIANTS = {"max": "max pooling", "avg": "average pooling", "blur": "MaxBlurPool"}
VARIANTS |= {"stride": "stride 2", "none": "no pooling", "bai11": "Lesson 11"}
K = lenet_runs["shift"]
runs = lenet_runs["runs"]
print(
    f"{len(runs)} networks × {len(lenet_runs['seeds'])} seeds, {lenet_runs['epochs']} epochs, "
    f"η = {num(lenet_runs['eta'], 1)}"
)


def spread(v, f=pct, d=1):
    return f"{f(min(v), d)}–{f(max(v), d)}"


def stats(rid):
    r = runs[rid]
    return {
        "test": [a[0] for a in r["acc"]],
        "chg1": [1 - s[1] for s in r["same"]],
        "acc4": [a[K] for a in r["acc"]],
    }


S = {rid: stats(rid) for rid in VARIANTS}
for rid, name in VARIANTS.items():
    s = S[rid]
    print(
        f"{name:>15}: {num(runs[rid]['params'], 0):>9} parameters · correct {spread(s['test'], d=2)} · "
        f"1-pixel shift changes {spread(s['chg1'])} · {K}-pixel shift still correct {spread(s['acc4'])}"
    )

The figure below plots two columns of the table. Each row is a network, with its name on the left; the three dots are
the three seeds, the horizontal bar joins the min to the max, and the vertical tick is the median. The max pooling
row is in blue. Left: the share of predictions that change under a 1-pixel shift (lower is more stable). Right: the
share still correct under a 4-pixel shift (higher is better).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
names = list(VARIANTS)
for ax, key, title in [
    (axes[0], "chg1", "Predictions changed by a 1-pixel shift (%)"),
    (axes[1], "acc4", f"Still correct after a {K}-pixel shift (%)"),
]:
    for row, rid in enumerate(names):
        v = 100 * np.array(S[rid][key])
        col = BLUE if rid == "max" else "black"
        ax.plot([v.min(), v.max()], [row, row], color=col, lw=2, alpha=0.6)
        ax.plot(v, [row] * 3, "o", color=col, ms=6, mfc="white" if rid != "max" else BLUE)
        ax.plot([np.median(v)] * 2, [row - 0.25, row + 0.25], color=col, lw=2)
    ax.set_title(title, fontsize=10)
    ax.grid(axis="x", alpha=0.3)
axes[0].set_yticks(range(len(names)), [VARIANTS[r] for r in names])
axes[0].invert_yaxis()
fmt_axes(fig, "x")
plt.tight_layout()
plt.show()

The cell below compares each network with the LeNet with **no pooling**, by the demo's rule: two networks differ
clearly only when their 3-seed ranges do not overlap.

In [ ]:
def overlap(a, b):
    return max(min(a), min(b)) <= min(max(a), max(b))


def verdict(a, b, high_good):
    if overlap(a, b):
        return "on a par (within the variation across seeds)"
    return "clearly better" if (np.median(a) > np.median(b)) == high_good else "clearly worse"


for rid in ["max", "avg", "blur", "stride"]:
    print(
        f"{VARIANTS[rid]:>15} vs no pooling: 1-pixel shift {verdict(S[rid]['chg1'], S['none']['chg1'], False)}; "
        f"{K}-pixel shift {verdict(S[rid]['acc4'], S['none']['acc4'], True)}"
    )
print(f"MaxBlurPool vs max pooling, {K}-pixel shift: {verdict(S['blur']['acc4'], S['max']['acc4'], True)}")
lo, hi = min(S["max"]["chg1"]), max(S["max"]["chg1"])
print(
    f"our short run (section 6): {pct(change1)} changed; "
    f"within max pooling's 3-seed range ({spread(S['max']['chg1'])}):",
    lo - 0.005 <= change1 <= hi + 0.005,
)
best4 = max(VARIANTS, key=lambda r: np.median(S[r]["acc4"]))
ratio = runs["none"]["params"] / runs["max"]["params"]
print(f"{K}-pixel shift, highest median: {VARIANTS[best4]} {pct(np.median(S[best4]['acc4']), 1)}")
print(f"no pooling / max pooling: {num(ratio, 1)} times as many parameters")

What to remember:
- **1-pixel shift:** every network changes some of its predictions, even the network with no pooling at all
  (1.8–2.2%). The three kinds of pooling are on a par with no pooling, within the variation across seeds; only
  stride 2 is clearly worse.
- **4-pixel shift:** all three kinds of pooling keep clearly more correct than no pooling, while stride 2 is clearly
  worse. Aggregating neighboring cells (taking the max, taking the mean, blurring) helps tolerate shifts partly;
  subsampling alone does not. MaxBlurPool has the highest median, but its seed range overlaps with max pooling's.
- **Size:** without pooling the map does not shrink, the first dense layer receives 9,216 values and the network has
  1,119,626 parameters, 18.1 times as many, and yet it does worse at large shifts.
- **No network is invariant:** under a 4-pixel shift, the best network (MaxBlurPool) is still correct on only 62.7%
  (median), whereas on unshifted images it gets 98.01–99.05% right.

To make a network tolerate shifts better, you have to teach it: data augmentation with shifted images (Lesson 11 did
this with an MLP), or anti-aliased subsampling like MaxBlurPool. Azulay & Weiss (2018) show that both solve the
problem only partly.

## 8 · Counting parameters and operations, layer by layer

The size formula from section 4 is all you need to count everything in a CNN without running it. For a $k \times k$
convolutional layer from $C_{\text{in}}$ to $C_{\text{out}}$ channels, with an $n_{\text{out}} \times n_{\text{out}}$
output map:
- **parameters:** $k^2 C_{\text{in}} C_{\text{out}} + C_{\text{out}}$ (the weights are shared across all positions,
  plus one bias per filter);
- **multiply-accumulate operations (MACs):** $n_{\text{out}}^2 \cdot k^2 C_{\text{in}} C_{\text{out}}$, because each
  weight is reused at every position.

A dense layer from $n_{\text{in}}$ to $n_{\text{out}}$ has $n_{\text{in}} n_{\text{out}} + n_{\text{out}}$ parameters
and $n_{\text{in}} n_{\text{out}}$ MACs: each weight is used only once. As in the demo, we count only multiplications
by a weight (pooling, ReLU and biases are not counted). The function `count()` below describes a network as a list of
layers, like the demo's “Build a CNN” card, and prints a table, layer by layer.

In [ ]:
def conv(k, s, p, c, groups=1, pairs=None):
    return {"t": "conv", "k": k, "s": s, "p": p, "c": c, "g": groups, "pairs": pairs}


def pool(k=2, s=2):
    return {"t": "pool", "k": k, "s": s, "p": 0}


def fc(c, fixed=False):
    return {"t": "fc", "c": c, "fixed": fixed}


# LeNet‑5's “subsampling”: sum a 2×2 window, multiply by a learned coefficient, add a bias (2 parameters per map)
SUB = {"t": "sub"}
BLUR = {"t": "blur"}  # MaxBlurPool: ⌈n/2⌉, no parameters
BN = {"t": "bn"}  # BatchNorm (Lesson 10): γ and β per channel, same size
GAP = {"t": "gap"}  # global average pooling: n × n × c → 1 × 1 × c


def count(n, c, layers, show=False):
    """Walk through the layers: output size, parameters, MACs. Returns a list of rows (kind, n, c, parameters, MACs)."""
    rows, flat = [], False
    for L in layers:
        if L["t"] == "fc":
            n_in = c if flat else n * n * c
            params, macs = (0 if L["fixed"] else n_in * L["c"] + L["c"]), n_in * L["c"]
            n, c, flat = 1, L["c"], True
        elif L["t"] == "bn":
            params, macs = 2 * c, 0
        elif L["t"] == "gap":
            n, params, macs = 1, 0, 0
        elif L["t"] in ("sub", "blur"):
            n = n // 2 if L["t"] == "sub" else -(-n // 2)
            params, macs = (2 * c, n * n * c) if L["t"] == "sub" else (0, 0)
        else:
            n = n_out(n, L["k"], L["s"], L["p"])
            params = macs = 0
            if L["t"] == "conv":
                per = L["k"] ** 2 * (L["pairs"] if L["pairs"] else c // L["g"] * L["c"])  # weights per position
                params, macs, c = per + L["c"], n * n * per, L["c"]
        rows.append((L["t"], n, c, params, macs))
        if show:
            shape = f"{c}" if flat else f"{n}×{n}×{c}"
            print(f"  {L['t']:<5} → {shape:<12} {num(params, 0):>12} parameters {num(macs, 0):>16} MACs")
    return rows


def totals(rows):
    P, M = sum(r[3] for r in rows), sum(r[4] for r in rows)
    Pd, Md = sum(r[3] for r in rows if r[0] == "fc"), sum(r[4] for r in rows if r[0] == "fc")
    return P, M, Pd / P, Md / M

First, the six MNIST networks of section 7. The parameter counts that `count()` computes must equal exactly those of
the networks the demo trained (and the `n_params` of the network we wrote ourselves in section 6):

In [ ]:
HEAD = [fc(120), fc(84), fc(10)]
MNIST_NETS = {
    "max": [conv(5, 1, 2, 6), pool(), conv(5, 1, 0, 16), pool(), *HEAD],
    "avg": [conv(5, 1, 2, 6), pool(), conv(5, 1, 0, 16), pool(), *HEAD],
    "blur": [conv(5, 1, 2, 6), BLUR, conv(5, 1, 0, 16), BLUR, *HEAD],
    "stride": [conv(5, 2, 2, 6), conv(5, 2, 0, 16), *HEAD],
    "none": [conv(5, 1, 2, 6), conv(5, 1, 0, 16), *HEAD],
    "bai11": [conv(3, 1, 0, 8), pool(), fc(10)],
}
print("LeNet with max pooling, layer by layer:")
count(28, 1, MNIST_NETS["max"], show=True)
for rid, layers in MNIST_NETS.items():
    P_, M_, pd, md = totals(count(28, 1, layers))
    print(
        f"{VARIANTS[rid]:>15}: {num(P_, 0):>9} parameters, {num(M_, 0):>10} MACs · dense: {pct(pd, 1)} of parameters, "
        f"{pct(md, 1)} of MACs"
    )
    assert P_ == runs[rid]["params"], "differs from the networks the demo trained"
assert totals(count(28, 1, MNIST_NETS["max"]))[0] == n_params == 61706
assert totals(count(28, 1, MNIST_NETS["none"]))[0] == 1119626
_, n_none, c_none, _, _ = count(28, 1, MNIST_NETS["none"])[1]
print(f"no pooling: the first dense layer receives {n_none}×{n_none}×{c_none} = {num(n_none**2 * c_none, 0)} values")

Stride 2 has exactly the same number of parameters as the max pooling version (the same filters, only the shrinking
differs) but far fewer operations, because the convolution is computed at only one position in four. Dropping pooling
does the opposite: the convolutional layers keep the same number of parameters, but the first dense layer has to
connect to 24×24×16 values, and the network swells to 18.1 times the size.

### LeNet‑5 (LeCun et al., 1998)

The original differs from the demo's version in five places: a 32×32 input image (the 28×28 digits are padded) and
convolutions without padding; shrinking by **subsampling** (sum 4 cells, multiply by a learned coefficient, add a
bias: 2 parameters per map); a C3 layer that connects only 60 pairs of maps (each map of C3 sees 3, 4 or 6 of S2's 6
maps; Table 1 of the paper); a scaled tanh activation, $f(a) = 1.7159 \tanh(Sa)$, not ReLU; and an output layer of 10
RBF units whose parameter vectors are set by hand and kept fixed (“at least initially”), instead of a softmax. The
paper gives the number of trainable parameters of each layer, so we can compare layer by layer.

In [ ]:
C3_PAIRS = 6 * 3 + 6 * 4 + 3 * 4 + 1 * 6  # 6 maps see 3 of S2's maps, 6 + 3 maps see 4, 1 map sees all 6
LENET5 = [conv(5, 1, 0, 6), SUB, conv(5, 1, 0, 16, pairs=C3_PAIRS), SUB, conv(5, 1, 0, 120), fc(84), fc(10, fixed=True)]
names5 = ["C1", "S2", "C3", "S4", "C5", "F6", "out (RBF)"]
print(f"C3 connects {C3_PAIRS} pairs of maps")
rows5 = count(32, 1, LENET5, show=True)
paper = [156, 12, 1516, 32, 48120, 10164, 0]  # LeCun et al. (1998), section II‑B
for name, r, want in zip(names5, rows5, paper):
    print(f"{name:>9}: {num(r[3], 0):>6} parameters (paper: {num(want, 0)})")
P5 = totals(rows5)[0]
print(f"total: {num(P5, 0)} trainable parameters; a fully connected C3 would have {num(5 * 5 * 6 * 16 + 16, 0)}")
assert [r[3] for r in rows5] == paper and P5 == 60000 and 5 * 5 * 6 * 16 + 16 == 2416

The total is exactly 60,000, the number the paper gives for LeNet‑5 (“345 308 connections, but only 60 000 trainable
free parameters”). If C3 connected all 6 × 16 pairs, that layer alone would have 2,416 parameters instead of 1,516.

### AlexNet (Krizhevsky, Sutskever & Hinton, 2012)

Five convolutional layers (11×11 with stride 4, 5×5, three 3×3 layers), 3×3 max pooling with stride 2 (the windows
overlap), then three dense layers 4096 → 4096 → 1000. The network is split across **2 GPUs** (GTX 580, 3 GB): in
convolutional layers 2, 4 and 5, each half of the filters connects only to the half of the channels on the same GPU
(`groups=2`), so these layers have half the weights of a fully connected version. The input image is taken as
227×227, as in section 4.

### 🤔 Predict first
AlexNet has about 60 million parameters. What percentage of the **parameters** do the three dense layers at the end
hold? And what percentage of the **multiply-accumulate operations**: about the same, or something entirely
different?

In [ ]:
ALEXNET = [conv(11, 4, 0, 96), pool(3, 2), conv(5, 1, 2, 256, groups=2), pool(3, 2), conv(3, 1, 1, 384)]
ALEXNET += [conv(3, 1, 1, 384, groups=2), conv(3, 1, 1, 256, groups=2), pool(3, 2), fc(4096), fc(4096), fc(1000)]
rows_a = count(227, 3, ALEXNET, show=True)
Pa, Ma, pa, ma = totals(rows_a)
print(f"AlexNet: {num(Pa, 0)} parameters, {num(Ma, 0)} MACs per image")
print(f"3 dense layers: {pct(pa, 1)} of the parameters but only {pct(ma, 1)} of the multiply-accumulates")
ungrouped = totals(count(227, 3, [{**L, "g": 1} if L["t"] == "conv" else L for L in ALEXNET]))[0]
print(f"without the 2-GPU split: {num(ungrouped, 0)} parameters")
neurons = [r[1] * r[1] * r[2] for r in rows_a if r[0] == "conv"]
paper_neurons = [253440, 186624, 64896, 64896, 43264]  # the caption of the paper's Figure 2
print("neurons in the 5 convolutional layers:", " – ".join(num(v, 0) for v in neurons))
print("the paper (Figure 2):                 ", " – ".join(num(v, 0) for v in paper_neurons))
print(f"each weight of layer 1 is used at {rows_a[0][1]} × {rows_a[0][1]} = {num(rows_a[0][1] ** 2, 0)} positions")
assert Pa == 60965224 and ungrouped == 62378344 and round(100 * pa) == 96 and round(100 * ma) == 8
assert neurons[1:] == paper_neurons[1:]  # matches the paper's Figure 2, except the first layer (see below)

The three dense layers hold 96% of the parameters but do only about 8% of the computation. A dense weight is used
only once per image; a convolutional weight is reused at every position of the map (55 × 55 = 3,025 times in layer
1). So parameters and operations are two different measures of a network's “size”: the parameters determine the
memory needed to store the network, the operations determine its running time.

The total of 60,965,224 is for the 2-GPU version, and matches the paper's “60 million parameters”. The neuron counts
of layers 2–5 match the sequence “253,440–186,624–64,896–64,896–43,264” in the caption of Figure 2; only for the
first layer, 55 × 55 × 96 = 290,400, not 253,440, an inconsistency in the paper.

AlexNet won ILSVRC‑2012 with a top‑5 error of 15.3%, versus 26.2% for the second-place team. The 15.3% is for an
ensemble of 7 CNNs (5 networks like the one above and 2 networks with an extra convolutional layer, pretrained on
ImageNet Fall 2011); a single network has 18.2% on the validation set (Table 2 of the paper). Besides GPUs, the paper
emphasizes ReLU (a 4-layer network with ReLU reaches a 25% training error on CIFAR‑10 6 times faster than with tanh)
and dropout in the first two dense layers.

### VGG‑16 (Simonyan & Zisserman, 2014)

Only two kinds of layer: 3×3 convolution with stride 1 and padding 1 (same size), and 2×2 max pooling with stride 2.
Five blocks of 2‑2‑3‑3‑3 convolutional layers with 64‑128‑256‑512‑512 channels, then dense 4096 → 4096 → 1000: in
total 13 + 3 = 16 weight layers.

In [ ]:
VGG16 = []
for n_conv, width in zip((2, 2, 3, 3, 3), (64, 128, 256, 512, 512)):
    VGG16 += [conv(3, 1, 1, width)] * n_conv + [pool()]
VGG16 += [fc(4096), fc(4096), fc(1000)]
rows_v = count(224, 3, VGG16, show=True)
Pv, Mv, pv, mv = totals(rows_v)
print(
    f"VGG‑16: {num(Pv, 0)} parameters, {num(Mv / 1e9, 2)} billion MACs · dense: {pct(pv, 1)} of parameters, "
    f"{pct(mv, 1)} of MACs"
)
same_cost = [r[4] for r in rows_v if r[0] == "conv" and r[1] >= 28]
print(f"MACs of the 3×3 layers in blocks 1–4: {' · '.join(num(m / 1e9, 2) for m in same_cost)} (billion)")
first_fc = next(r for r in rows_v if r[0] == "fc")
print(
    f"the first dense layer alone (7×7×512 → 4096): {num(first_fc[3], 0)} parameters = {pct(first_fc[3] / Pv, 1)} "
    "of the whole network"
)
# torchvision: vgg16 num_params 138357544, GFLOPS 15.47 (this number is really the MAC count); the paper, Table 2:
# 138 million
assert Pv == 138357544 and round(Mv / 1e9, 2) == 15.47
assert len(set(same_cost[1:2] + same_cost[3:4] + same_cost[5:7] + same_cost[8:10])) == 1

138,357,544 parameters: exactly the number for torchvision's VGG‑16, and the “138” million in Table 2 of the paper.
One 224×224 image costs 15.47 billion MACs (torchvision lists “15.47 GFLOPS”, which actually counts MACs). The first
dense layer alone holds 74.3% of the whole network's parameters, while the dense layers do only 0.8% of the
computation.

The layer-by-layer table shows the common pattern of classic CNNs: **the spatial size shrinks while the number of
channels grows**. At each pooling, the number of positions drops 4 times while the number of channels doubles (the
weights per position grow 4 times), so every 3×3 layer in blocks 1–4 with the same number of input and output
channels costs exactly 1.85 billion MACs. The first layer of blocks 2–4 (with half as many input channels) costs
half, 0.92 billion; block 5 keeps 512 channels, so it is cheaper.

VGG placed 2nd in the classification task of ILSVRC‑2014 with a top‑5 error of 7.3% (an ensemble of 7 networks);
after the competition, an ensemble of the 2 best networks brought it down to 6.8% (the paper, section 4.5). The paper
tries networks from 11 to 19 weight layers: the error decreases with depth and “saturates” at 19 layers.

The figure below summarizes the counts: for each network, one bar for the parameters and one for the
multiply-accumulates, split into the share of the convolutional layers (blue, solid) and of the dense layers (orange,
hatched), with the dense percentage written next to it.

In [ ]:
NETS = {
    "LeNet (MNIST, section 6)": count(28, 1, MNIST_NETS["max"]),
    "LeNet‑5 (1998)": rows5,
    "AlexNet (2012)": rows_a,
    "VGG‑16 (2014)": rows_v,
}
fig, ax = plt.subplots(figsize=(9, 3.8))
labels = []
for i, (name, rows) in enumerate(NETS.items()):
    _, _, p_d, m_d = totals(rows)
    for jj, (what, share) in enumerate([("parameters", p_d), ("multiply-accumulates", m_d)]):
        y = 2.6 * i + jj
        ax.barh(y, 100 * (1 - share), color=BLUE, height=0.8)
        ax.barh(y, 100 * share, left=100 * (1 - share), color=ORANGE, height=0.8, hatch="//", edgecolor="white", lw=0)
        ax.text(101, y, f"dense {pct(share, 1)}", va="center", fontsize=9)
        labels.append((y, f"{name} · {what}"))
ax.set_yticks([y for y, _ in labels], [t for _, t in labels], fontsize=9)
ax.invert_yaxis()
ax.set(xlim=(0, 125), xlabel="percent of the whole network (blue: convolution · hatched orange: dense)")
ax.set_xticks([0, 25, 50, 75, 100])
fmt_axes(fig, "x")
plt.tight_layout()
plt.show()

In every network with large dense layers, the parameters concentrate in the dense layers and the computation in the
convolutions. LeNet‑5 is the exception for parameters only because the paper calls C5 a convolutional layer (5×5 on a
5×5 map, so in effect fully connected); we follow that naming.

## 9 · Try changing things

The cells below rerun the same functions with different settings. You can edit the values and run them again.

**Change the architecture and count again.** Three changes: AlexNet with the 224×224 image the paper states (no
padding in layer 1); VGG‑16 with its three dense layers replaced by global average pooling (averaging each channel
over the whole 7×7 map, as ResNet does in Lesson 13) followed by one dense layer 512 → 1000; and VGG‑16 without its
last pooling.

In [ ]:
experiments = {
    "AlexNet, 224×224 image": (224, ALEXNET),
    "VGG‑16, GAP + dense 1000": (224, VGG16[:-4] + [pool(), GAP, fc(1000)]),
    "VGG‑16, no last pooling": (224, VGG16[:-4] + VGG16[-3:]),
}
for name, (n, layers) in experiments.items():
    rows = count(n, 3, layers)
    P_, M_, p_d, _ = totals(rows)
    k_fc = next(i for i, r in enumerate(rows) if r[0] == "fc")
    _, n_last, c_last, _, _ = rows[k_fc - 1]
    print(
        f"{name:<26}: {num(P_, 0):>12} parameters, {num(M_ / 1e9, 2):>5} billion MACs"
        f" · dense {pct(p_d, 1)} of parameters"
        f" · the first dense layer receives {n_last}×{n_last}×{c_last} = {num(n_last**2 * c_last, 0)} values,"
        f" with {num(rows[k_fc][3], 0)} parameters"
    )

Three notable results:
- **AlexNet with a 224 image and no padding:** layer 1 outputs 54×54, not 55×55. The later layers shrink
  accordingly, and the first dense layer receives only 5×5×256 = 6,400 values instead of 6×6×256 = 9,216: a different
  network, smaller than the one the paper describes (with padding 2 in layer 1, it is the 227 input's network again,
  as in section 4).
- **VGG‑16 with GAP:** dropping the three dense layers cuts the parameters from 138 million to about 15 million,
  while the computation stays almost the same, because the computation was in the convolutional layers all along.
  The networks after VGG (GoogLeNet, ResNet) all end with global average pooling.
- **VGG‑16 without the last pooling:** the map before the dense layers is still 14×14, so the first dense layer alone
  has more than 400 million parameters.

**Change the blur filter.** Zhang (2019) tries several filters before subsampling: $[1\ 1]$ (equivalent to average
pooling), $[1\ 2\ 1]$ (used above) and $[1\ 4\ 6\ 4\ 1]$. The cell below measures the change under a 1-pixel shift
again, as in section 3, for each filter.

In [ ]:
def maxblur(x, taps):
    """MaxBlurPool with a chosen filter: 2×2 max with stride 1, a taps ⊗ taps blur (normalized), then stride 2."""
    H, W = x.shape[1:3]
    xp = np.pad(x, [(0, 0), (0, 1), (0, 1)], mode="edge")
    m = np.maximum(np.maximum(xp[:, :H, :W], xp[:, 1:, :W]), np.maximum(xp[:, :H, 1:], xp[:, 1:, 1:]))
    w = np.array(taps, float) / sum(taps)
    lo_pad, hi_pad = (len(w) - 1) // 2, len(w) // 2
    mp = np.pad(m, [(0, 0), (lo_pad, hi_pad), (lo_pad, hi_pad)], mode="reflect")
    out = sum(w[i] * w[j] * mp[:, i : i + H, j : j + W] for i in range(len(w)) for j in range(len(w)))
    return out[:, ::2, ::2]


blur_change = {}
for taps in [(1,), (1, 1), (1, 2, 1), (1, 4, 6, 4, 1)]:
    base = maxblur(X2, taps)
    blur_change[taps] = rel_change(maxblur(shift(X2, 1, 0), taps), base)
    print(f"filter {vec(taps):<14}: 1-pixel shift → change {pct(blur_change[taps], 1)}")
assert abs(blur_change[(1,)] - sens["max pooling"][0]) < 1e-9  # the filter [1] = ordinary max pooling
assert abs(blur_change[(1, 2, 1)] - sens["MaxBlurPool"][0]) < 1e-9
assert blur_change[(1, 4, 6, 4, 1)] < blur_change[(1, 2, 1)] < blur_change[(1, 1)] < blur_change[(1,)]

The filter $[1]$ (no blur) gives exactly ordinary max pooling, and $[1\ 2\ 1]$ gives exactly the MaxBlurPool of
section 3. The wider the filter, the less the output changes under a shift, but also the blurrier it gets: the
network loses some detail. Zhang (2019) found that on ImageNet the accuracy still rises slightly with these filters,
but the network is still not fully stable under shifts.

**Retrain with a different pooling.** `train_lenet()` takes `kind="avg"` for average pooling, and a different
`seed`. The cell below trains average pooling for 200 steps and compares it with max pooling after the same number
of steps, on the first 2,000 test images so that it runs fast. Increase `steps` (600 as in section 6, or a whole
epoch: 938 steps) if you have more time.

In [ ]:
kind, seed, steps, n_eval = "avg", 0, 200, 2000
for k_ in (kind, "max"):
    P_ = train_lenet(k_, seed=seed, steps=steps)
    p0 = predict(P_, Xte[:n_eval], k_)
    ch = np.mean([(predict(P_, shift(Xte[:n_eval], *d), k_) != p0).mean() for d in DIRS])
    print(
        f"{k_}, seed {seed}, {steps} steps, {num(n_eval, 0)} test images: {pct((p0 == yte[:n_eval]).mean())} correct"
        f" · 1-pixel shift changes {pct(ch)}"
    )

After 200 steps both networks are still learning, so the accuracy is still lower than in section 6. One seed and one
short run cannot tell which pooling is more stable; to compare, look at the 3-seed ranges in section 7, or run this
cell with a few values of `seed`.

## 10 · VGG: deeper, then hitting the wall of depth

VGG showed that 16–19 layers beat 11 layers. So is adding layers always better? He et al. (2015) tested this on
CIFAR‑10 with **plain** networks (just a stack of layers, with no skip connections) of $6n + 2$ weight layers: a
first convolutional layer, three stages of $2n$ 3×3 layers each, then global average pooling (GAP) and a dense layer
of 10. Every convolutional layer is Conv → BatchNorm → ReLU.

The demo's `depth.py` script redoes that experiment, scaled down for MNIST: the first layer is a 3×3 convolution with
stride 2 (28 → 14), the three stages have 8, 16 and 32 channels, and the first layer of stages 2 and 3 has stride 2
(14 → 7 → 4). Depths of 8, 20, 32, 44 and 56 layers; SGD with batches of 64, 3 epochs on 55,000 images,
$\eta \in \{0.03,\ 0.1,\ 0.3\}$ × 3 seeds. Each depth's $\eta$ is chosen on 5,000 validation images (the end of the
training file); the training error is measured on the first 10,000 training images, after training.

Before reading the results, section 8's `count()` recomputes the number of parameters of each network; they must
match the trained networks.

In [ ]:
def plain_net(depth, widths=(8, 16, 32)):
    """A plain network in the style of He et al. (2015) for MNIST, as in depth.py: 6n + 2 weight layers."""
    per_stage = (depth - 2) // 3  # 2n 3×3 layers per stage
    layers = [conv(3, 2, 1, widths[0]), BN]
    for stage, w in enumerate(widths):
        for i in range(per_stage):
            layers += [conv(3, 2 if stage > 0 and i == 0 else 1, 1, w), BN]
    return layers + [GAP, fc(10)]


DEPTHS, ETAS = depth_runs["depths"], depth_runs["etas"]
cfg = {(r["depth"], r["eta"]): r["seeds"] for r in depth_runs["runs"]}
print(
    f"{len(DEPTHS)} depths × {len(ETAS)} values of η × {len(depth_runs['seeds'])} seeds, {depth_runs['epochs']} epochs "
    f"on {num(depth_runs['nTrain'], 0)} images; training error measured on the first "
    f"{num(depth_runs['nTrainEval'], 0)} training images"
)
for d in DEPTHS:
    rows_d = count(28, 1, plain_net(d))
    sizes = sorted({r[1] for r in rows_d if r[0] == "conv"}, reverse=True)
    n_layers = sum(r[0] in ("conv", "fc") for r in rows_d)
    P_d = totals(rows_d)[0]
    print(f"{d:>2} layers: {n_layers} weight layers, maps {' → '.join(map(str, sizes))}, {num(P_d, 0):>7} parameters")
    assert n_layers == d and P_d == cfg[(d, ETAS[0])][0]["params"], "differs from the networks the demo trained"

### 🤔 Predict first
The 56-layer network has more than three times the parameters of the 20-layer one, and contains it: make the 36
extra layers the identity and you get exactly the 20-layer network. After the same 3 epochs, each network with its
best $\eta$, will the 56-layer network's error on the **training data** itself be lower than, equal to, or higher
than the 20-layer network's?

In [ ]:
def val_median(d, e):
    return np.median([s.get("val", 1.0) for s in cfg[(d, e)]])  # a diverged run (NaN) = 100% error, as in the demo


best_eta = {d: min(ETAS, key=lambda e: val_median(d, e)) for d in DEPTHS}
err = {d: {k: [s[k] for s in cfg[(d, best_eta[d])]] for k in ("train", "val", "test")} for d in DEPTHS}
for d in DEPTHS:
    e = err[d]
    print(
        f"{d:>2} layers, η = {num(best_eta[d], 2)}: training error {spread(e['train'], d=2):<13} · validation "
        f"{spread(e['val'], d=2):<13} · test {spread(e['test'], d=2)}"
    )
best_d = min(DEPTHS, key=lambda d: np.median(err[d]["train"]))
print(
    f"lowest training error: {best_d} layers; 56 layers vs {best_d} layers: "
    f"{verdict(err[56]['train'], err[best_d]['train'], False)}"
)

The 56-layer network has a clearly higher error on **the training data itself** than the 20-layer network: the
3-seed ranges of the two networks do not overlap. The error falls from 8 to 20 layers, then rises steadily from 20 to
56 layers. The figure below plots the table above: for each depth, the training error (blue circles, solid line) and
the validation error (orange squares, dashed line); the small markers are the individual seeds, and the line joins
the medians.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
for key, label, col, mk, ls, dx in [
    ("train", "training error (10,000 training images)", BLUE, "o", "-", -0.6),
    ("val", "validation error (5,000 images)", ORANGE, "s", "--", 0.6),
]:
    med = [100 * np.median(err[d][key]) for d in DEPTHS]
    ax.plot(DEPTHS, med, color=col, ls=ls, marker=mk, ms=8, lw=2, label=label)
    for d in DEPTHS:
        ax.plot([d + dx] * 3, 100 * np.array(err[d][key]), mk, color=col, ms=4, mfc="white")
ax.set(xlabel="number of weight layers", ylabel="error after 3 epochs (%)", xticks=DEPTHS, ylim=(0, None))
ax.set_title("Plain networks on MNIST: from 20 layers on, more layers raise the training error (3 seeds)", fontsize=10)
ax.grid(axis="y", alpha=0.3)
ax.legend(frameon=False, fontsize=9)
fmt_axes(fig)
plt.tight_layout()
plt.show()

This is not **overfitting** (Lesson 08): overfitting means a low training error but a high validation error. Here the
validation error tracks the training error closely, and the deep network is worse on the very images it has learned.
He et al. call this the **degradation problem**: “adding more layers to a suitably deep model leads to higher
training error”.

Is it because of $\eta$? The cell below compares 56 layers with 20 layers at **each** value of $\eta$, with no
selection at all.

In [ ]:
for e in ETAS:
    a, b = [s["train"] for s in cfg[(56, e)]], [s["train"] for s in cfg[(20, e)]]
    print(
        f"η = {num(e, 2)}: 56 layers {spread(a, d=2):<13} · 20 layers {spread(b, d=2):<13} → 56 layers "
        f"{verdict(a, b, False)}"
    )

At all three learning rates, the 56-layer network is clearly worse. The figure below plots the training error
*during* training (the mean over each 100 batches, the median of 3 seeds, a log-scale vertical axis). The 20-layer
(blue, solid) and 56-layer (orange, dashed) networks are in color; the other depths are thin gray lines, with the
number of layers written at the end of each line.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
steps_axis = None
for d in DEPTHS:
    curves = np.array([s["curve"] for s in cfg[(d, best_eta[d])]])  # (3 seeds, points, [loss, error])
    med = np.median(curves[:, :, 1], axis=0)
    steps_axis = depth_runs["log"] * np.arange(1, len(med) + 1)
    style = {20: (BLUE, "-", 2.2), 56: (ORANGE, "--", 2.2)}.get(d, (GREY, "-", 1))
    ax.plot(steps_axis, 100 * med, color=style[0], ls=style[1], lw=style[2])
    ax.annotate(
        f"{d} layers",
        (steps_axis[-1], 100 * med[-1]),
        xytext=(4, 0),
        textcoords="offset points",
        fontsize=8,
        va="center",
        color="black" if d in (20, 56) else GREY,
    )
ax.set(yscale="log", xlabel="SGD step", ylabel="running training error (%)", xlim=(0, steps_axis[-1] * 1.12))
ax.set_title("Training error over 3 epochs (median of 3 seeds, best η for each depth)", fontsize=10)
ax.grid(alpha=0.3, which="both")
fmt_axes(fig)
plt.tight_layout()
plt.show()

The 56-layer network is worse **throughout** training, not only at the end, as in Figure 1 of He et al. Maybe it just
needs to train longer? The demo also trains 20 and 56 layers for 10 epochs (seed 0 only, $\eta$ 0.03 and 0.1):

In [ ]:
long = depth_runs["long"]
best_long = {d: min((r for r in long if r["depth"] == d), key=lambda r: r["val"]) for d in (20, 56)}
for d, r in best_long.items():
    print(
        f"{depth_runs['longEpochs']} epochs, {d} layers (η = {num(r['eta'], 2)}, chosen on validation): training error "
        f"{pct(r['train'])} · validation {pct(r['val'])} · test {pct(r['test'])}"
    )
print(f"3 epochs, 20 layers (median of 3 seeds): training error {pct(np.median(err[20]['train']))}")

After 10 epochs, the 56-layer network still has a 3.06% training error, higher even than the 20-layer network after
only 3 epochs. This is only one seed, so it is an extra check rather than a conclusion, but it shows that the gap is
not just a matter of not having trained long enough.

Why does the deeper network learn worse? Not because it **cannot** do well: take the trained 20-layer network, insert
36 layers that are the identity, and you get a 56-layer network with exactly the 20-layer network's error. He et al.
write: “There exists a solution by construction to the deeper model”. Nor is it simply vanishing gradients (Lesson
09): every layer already has BatchNorm, and He et al. verified that the gradients in their plain networks have
normal magnitudes. The problem is **optimization**: SGD does not find the good solution that the deep network
certainly has, because a stack of nonlinear layers has a very hard time learning the identity exactly.

Lesson 13 fixes this with a very small change: let each pair of layers compute $x + F(x)$ instead of $F(x)$. Then
the identity only needs $F = 0$, which is very easy to learn, and adding layers no longer makes the network learn
worse. On CIFAR‑10, He et al. report a test error of 8.75% for the 20-layer ResNet and 6.97% for the 56-layer ResNet
(Table 6).

## Summary
- 2×2 pooling with stride 2 halves each side and has no parameters. Shift the image by 2 pixels and the output
  shifts by exactly 1 cell; shift it by 1 pixel and no grid can follow, and max pooling's 14×14 map changes by 46.5%.
- $n_{\text{out}} = \lfloor (n_{\text{in}} + 2p - k)/s \rfloor + 1$, checked on 3,644 combinations; the remainder is
  dropped, so AlexNet needs a 227 image, not 224.
- Receptive field: $r \leftarrow r + (k-1)j$, $j \leftarrow js$. LeNet sees 16×16, VGG‑16 sees 212×212 with 3×3
  filters alone; two 3×3 layers see as much as one 5×5 layer, with $18C^2$ instead of $25C^2$ weights.
- Max pooling does not make the network invariant: a 1-pixel shift changes 1.3–3.9% of LeNet's predictions (3
  seeds), on a par with the network with no pooling; at a 4-pixel shift, pooling clearly helps (50.3–58.6% versus
  47.1–48.8%), but even the best network keeps only 56.7–63.3%.
- Counting layer by layer matches the published numbers: exactly 60,000 parameters for LeNet‑5 and 138,357,544 for
  VGG‑16 (torchvision); AlexNet has 60,965,224, matching the paper's “60 million”. Parameters concentrate in the dense
  layers (AlexNet 96%), computation in the convolutions (dense only 8%).
- A deeper plain network learns worse: 56 layers have a training error of 5.78–12.84%, 20 layers 0.87–1.10%. Not
  overfitting, but difficulty optimizing (the degradation problem).

## Review questions
1. Why does shifting the image by 2 pixels shift the output of 2×2 max pooling by exactly 1 cell, while shifting it
   by 1 pixel changes the output entirely?
2. A 7×7 convolutional layer with stride 2 and padding 3 takes a 224×224 map. What size is the output? How many rows
   are dropped?
3. Two 3×3 layers with stride 1, then 2×2 max pooling with stride 2, then one 3×3 layer with stride 1: what is the
   receptive field?
4. In AlexNet, why do the three dense layers hold 96% of the parameters while doing only about 8% of the
   computation?
5. The LeNet with max pooling changes 1.3–3.9% of its predictions under a 1-pixel shift, the LeNet without pooling
   1.8–2.2%. Can we conclude which network is more stable?
6. The 56-layer plain network has a higher training error than the 20-layer one. Why is that not overfitting, and
   why is it surprising?

<details><summary>Hints and answers</summary>

1. Max pooling = take the max over every window (which shifts with the input), then keep one cell in two. With a
   2-pixel shift, the kept cells are still the same windows, just shifted by 1 cell; with a 1-pixel shift, the kept
   cells are entirely different windows.
2. $\lfloor (224 + 6 - 7)/2 \rfloor + 1 = \lfloor 111.5 \rfloor + 1 = 112$; $223 \bmod 2 = 1$ row is dropped (the
   first layer of ResNet).
3. $r = 1 + 2 + 2 = 5$, $j = 1$; pooling: $r = 5 + 1 = 6$, $j = 2$; the last layer: $r = 6 + 2 \cdot 2 = 10$. A 10×10
   region.
4. A dense weight is used only once per image; a convolutional weight is used at every position of the map (3,025
   positions in layer 1), so few parameters but a lot of computation.
5. No: the two 3-seed ranges overlap, so the difference is within the variation across seeds.
6. Overfitting means a low training error but a high validation error; here the deep network is worse on the training
   data itself. It is surprising because the 56-layer network contains the 20-layer one (with the extra layers as the
   identity), so it should do at least as well; SGD does not find that solution.
</details>

## Further reading
- Y. LeCun, L. Bottou, Y. Bengio & P. Haffner (1998), [Gradient-Based Learning Applied to Document
  Recognition](https://doi.org/10.1109/5.726791): LeNet‑5; section II‑B gives the number of parameters of each
  layer.
- A. Krizhevsky, I. Sutskever & G. Hinton (2012), [ImageNet Classification with Deep Convolutional Neural
  Networks](https://papers.nips.cc/paper_files/paper/2012/hash/c399862d3b9d6b76c8436e924a68c45b-Abstract.html):
  AlexNet; Table 2 has the errors of 1, 5 and 7 networks.
- K. Simonyan & A. Zisserman (2014), [Very Deep Convolutional Networks for Large-Scale Image
  Recognition](https://arxiv.org/abs/1409.1556): VGG; section 2.3 on stacking 3×3 layers.
- K. He, X. Zhang, S. Ren & J. Sun (2015), [Deep Residual Learning for Image Recognition](https://arxiv.org/abs/1512.03385):
  Figure 1 and section 4.1 on the degradation problem.
- A. Araujo, W. Norris & J. Sim (2019), [Computing Receptive Fields of Convolutional Neural
  Networks](https://distill.pub/2019/computing-receptive-fields/), Distill: the receptive field formula and the table
  of networks.
- W. Luo, Y. Li, R. Urtasun & R. Zemel (2016), [Understanding the Effective Receptive Field in Deep Convolutional
  Neural Networks](https://arxiv.org/abs/1701.04128).
- R. Zhang (2019), [Making Convolutional Networks Shift-Invariant Again](https://arxiv.org/abs/1904.11486): MaxBlurPool.
- A. Azulay & Y. Weiss (2018), [Why do deep convolutional networks generalize so poorly to small image
  transformations?](https://arxiv.org/abs/1805.12177)
- CS231n, [Convolutional Neural Networks](https://cs231n.github.io/convolutional-networks/): the size formula, the
  table of VGG‑16's memory and parameters, and the note about AlexNet's 227 image.

**Next lesson:** [Lesson 13 · Skip connections & ResNet](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/resnet/):
add $x$ to the output of each pair of layers, and deep networks no longer learn worse than shallow ones.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")